# Trenowanie modeli

In [ ]:
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

## Przygotowanie danych do treningu

In [2]:
import os, json, random, shutil, cv2
from pathlib import Path
import yaml

# Kategorie (foldery) – pobierz z nagrań lub metadata
categories = [
    "city_daylight",
    "city_daylight_rain",
    "city_night",
    "city_night_rain",
    "highway_daylight",
    "highway_daylight_rain",
    "highway_night",
    "highway_night_rain",
    "mountain_roads_daylight",
    "mountain_roads_daylight_rain",
    "mountain_roads_night",
    "mountain_roads_night_rain",
    "roadworks_daylight",
    "roadworks_daylight_rain",
    "roadworks_night",
    "roadworks_night_rain",
]

# Ścieżki
VIDEO_BASE = Path(r"D:\MGR\Master\recordings")
META_BASE = Path("../data/metadata")
DATASET_ALL = Path("../data/dataset")
IMAGES_ALL = DATASET_ALL / "images" / "all"
LABELS_ALL = DATASET_ALL / "labels" / "all"

# Tworzymy katalogi
# for p in [IMAGES_ALL, LABELS_ALL]:
#     p.mkdir(parents=True, exist_ok=True)
# print("Ścieżki utworzone.")

In [ ]:
# Ekstrakcja pełnych klatek + adnotacji


def build_fullframe_dataset(video_base, meta_base, images_out, labels_out):
    for cat in categories:
        mp4_dir = video_base / cat / "mp4"
        md_dir = meta_base / cat
        if not mp4_dir.is_dir() or not md_dir.is_dir():
            print(f"[WARN] Brak folderu mp4/ lub metadata/ dla {cat}, pomijam.")
            continue

        # Przejdź po wszystkich plikach json w md_dir
        for json_file in md_dir.glob("*.json"):
            base_name = json_file.stem  # np. "20240308_143701F_c_d"
            # odpowiadający plik mp4
            mp4_name = base_name.replace("_", " ").upper() + ".mp4"
            video_path = mp4_dir / mp4_name
            if not video_path.exists():
                print(f"[WARN] Brak wideo {mp4_name} w {mp4_dir}")
                continue

            # Wczytujemy metadane
            with open(json_file, "r", encoding="utf-8") as f:
                metadata = json.load(f)["metadata"]

            cap = cv2.VideoCapture(str(video_path))
            fps = cap.get(cv2.CAP_PROP_FPS)
            ih, iw = None, None

            # Dla każdej dziury
            for key, entry in metadata.items():
                t = entry["z"][0]
                frame_no = int(t * fps)
                cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)
                ret, frame = cap.read()
                if not ret:
                    continue
                ih, iw = frame.shape[:2]

                # bbox z metadanych
                x, y, w, h = entry["xy"][1:]
                x1, y1 = x, y
                x2, y2 = x + w, y + h

                # nazwa pliku wynikowego
                fname = f"{cat}_{frame_no}_{key}.jpg"
                img_out = images_out / fname
                lbl_out = labels_out / fname.replace(".jpg", ".txt")

                # zapis ramki
                cv2.imwrite(str(img_out), frame)
                # zapis etykiety YOLO
                xc = (x1 + x2) / 2 / iw
                yc = (y1 + y2) / 2 / ih
                wr = (x2 - x1) / iw
                hr = (y2 - y1) / ih
                with open(lbl_out, "w") as l:
                    l.write(f"0 {xc:.6f} {yc:.6f} {wr:.6f} {hr:.6f}\n")

            cap.release()


# Uruchamiamy
build_fullframe_dataset(VIDEO_BASE, META_BASE, IMAGES_ALL, LABELS_ALL)
print("Ekstrakcja zakończona (poprawione ścieżki).")

Ekstrakcja zakończona (poprawione ścieżki).


In [ ]:
# Stratyfikowany split na poziomie obrazów

# Przygotuj foldery (jeśli jeszcze nie istnieją)
TRAIN_IM = DATASET_ALL / "images" / "train"
VAL_IM = DATASET_ALL / "images" / "val"
TRAIN_LB = DATASET_ALL / "labels" / "train"
VAL_LB = DATASET_ALL / "labels" / "val"
for p in [TRAIN_IM, VAL_IM, TRAIN_LB, VAL_LB]:
    p.mkdir(parents=True, exist_ok=True)


# Funkcja do pobrania wszystkich obrazów danej kategorii
def images_for_category(cat):
    return list(IMAGES_ALL.glob(f"{cat}_*.jpg"))


# Split 80/20 wewnątrz każdej kategorii
random.seed(42)
for cat in categories:
    imgs = images_for_category(cat)
    random.shuffle(imgs)
    n_train = int(0.8 * len(imgs))
    train_imgs = imgs[:n_train]
    val_imgs = imgs[n_train:]

    # Kopiujemy
    for img in train_imgs:
        shutil.copy(img, TRAIN_IM / img.name)
        lbl = LABELS_ALL / img.with_suffix(".txt").name
        if lbl.exists():
            shutil.copy(lbl, TRAIN_LB / lbl.name)

    for img in val_imgs:
        shutil.copy(img, VAL_IM / img.name)
        lbl = LABELS_ALL / img.with_suffix(".txt").name
        if lbl.exists():
            shutil.copy(lbl, VAL_LB / lbl.name)

    print(f"{cat}: train {len(train_imgs)}, val {len(val_imgs)}")

# Podsumowanie
print("Łącznie train images:", len(list(TRAIN_IM.glob("*.jpg"))))
print("Łącznie val   images:", len(list(VAL_IM.glob("*.jpg"))))

city_daylight: train 5621, val 1406
city_daylight_rain: train 766, val 192
city_night: train 1041, val 261
city_night_rain: train 302, val 76
highway_daylight: train 1383, val 346
highway_daylight_rain: train 268, val 68
highway_night: train 224, val 56
highway_night_rain: train 33, val 9
mountain_roads_daylight: train 776, val 194
mountain_roads_daylight_rain: train 120, val 31
mountain_roads_night: train 48, val 13
mountain_roads_night_rain: train 0, val 0
roadworks_daylight: train 865, val 217
roadworks_daylight_rain: train 88, val 22
roadworks_night: train 199, val 50
roadworks_night_rain: train 46, val 12
Łącznie train images: 10483
Łącznie val   images: 2866


In [ ]:
cfg = {
    "train": str((DATASET_ALL / "images" / "train").resolve()),
    "val": str((DATASET_ALL / "images" / "val").resolve()),
    "nc": 1,
    "names": ["pothole"],
}
with open("../dataset.yaml", "w") as f:
    yaml.dump(cfg, f, sort_keys=False)
print("dataset.yaml:", cfg)

dataset.yaml: {'train': 'C:\\Users\\adamp\\OneDrive\\Pulpit\\MGR\\Master\\CarVision_MGR\\data\\dataset\\images\\train', 'val': 'C:\\Users\\adamp\\OneDrive\\Pulpit\\MGR\\Master\\CarVision_MGR\\data\\dataset\\images\\val', 'nc': 1, 'names': ['pothole']}


## YOLOv5s

In [9]:
!git clone https://github.com/ultralytics/yolov5.git

Cloning into 'yolov5'...


In [10]:
# Po sklonowaniu — dodaj yolov5 do sys.path
import sys
from pathlib import Path

yolo_dir = Path.cwd() / "yolov5"
sys.path.append(str(yolo_dir))

print("YOLOv5 repo dodane do PATH:", yolo_dir.exists())

YOLOv5 repo dodane do PATH: True


In [12]:
!pip install -r yolov5/requirements.txt

In [ ]:
import torch

print(torch.cuda.is_available())  # powinno być True
print(torch.cuda.device_count())  # >0

True
1


In [ ]:
import yolov5

In [ ]:
import torch

# 1) Załaduj model z Torch Hub
model = torch.hub.load("ultralytics/yolov5", "yolov5s", pretrained=True).eval().cuda()


# 2) Funkcja testująca forward-pass z ogólnym przechwyceniem błędów
def can_run_bs(bs):
    try:
        x = torch.zeros(bs, 3, 640, 640, device="cuda")
        with torch.no_grad():
            _ = model(x)
        torch.cuda.empty_cache()
        return True
    except RuntimeError:
        # dowolny błąd podczas forward-pass uznajemy za za dużo
        torch.cuda.empty_cache()
        return False


# 3) Binary-search batch_size
low, high, best = 1, 512, 0  # dajemy większy górny limit, ale kod go obniży
while low <= high:
    mid = (low + high) // 2
    print(f"Testing batch_size = {mid}...", end="")
    if can_run_bs(mid):
        print("OK")
        best, low = mid, mid + 1
    else:
        print("FAIL")
        high = mid - 1

print(f"\n→ Maksymalny batch_size bez błędu: {best}")

Using cache found in C:\Users\adamp/.cache\torch\hub\ultralytics_yolov5_master
YOLOv5  2025-7-31 Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 4GB Laptop GPU, 4096MiB)

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 
C:\Users\adamp/.cache\torch\hub\ultralytics_yolov5_master\models\common.py:879: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):


Testing batch_size = 256...OK
Testing batch_size = 384...

C:\Users\adamp/.cache\torch\hub\ultralytics_yolov5_master\models\common.py:879: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):


RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [3]:
!python yolov5/train.py --help

usage: train.py [-h] [--weights WEIGHTS] [--cfg CFG] [--data DATA] [--hyp HYP]
                [--epochs EPOCHS] [--batch-size BATCH_SIZE] [--imgsz IMGSZ]
                [--rect] [--resume [RESUME]] [--nosave] [--noval]
                [--noautoanchor] [--noplots] [--evolve [EVOLVE]]
                [--evolve_population EVOLVE_POPULATION]
                [--resume_evolve RESUME_EVOLVE] [--bucket BUCKET]
                [--cache [CACHE]] [--image-weights] [--device DEVICE]
                [--multi-scale] [--single-cls] [--optimizer {SGD,Adam,AdamW}]
                [--sync-bn] [--workers WORKERS] [--project PROJECT]
                [--name NAME] [--exist-ok] [--quad] [--cos-lr]
                [--label-smoothing LABEL_SMOOTHING] [--patience PATIENCE]
                [--freeze FREEZE [FREEZE ...]] [--save-period SAVE_PERIOD]
                [--seed SEED] [--local_rank LOCAL_RANK] [--entity ENTITY]
                [--upload_dataset [UPLOAD_DATASET]]
                [--bbox_interval BBOX_

#### Trening
python yolov5/train.py  --imgsz 640  --batch-size 32  --epochs 25  --data ../dataset.yaml  --cfg yolov5/models/yolov5s.yaml  --weights yolov5s.pt  --cache disk  --workers 12  --name pothole_yolov5s_optimal  --exist-ok

#### Walidacja
python yolov5/val.py --weights yolov5/runs/train/pothole_yolov5s_optimal/weights/best.pt --data ../dataset.yaml --imgsz 640

## YOLOv8n

In [9]:
# Klonujemy YOLOv8 (ultralytics/ultralytics)
# !git clone https://github.com/ultralytics/ultralytics.git
!pip install -r notebooks/ultralytics/requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'notebooks/ultralytics/requirements.txt'


In [10]:
import sys
from pathlib import Path

# Dodajemy yolov8 do PATH
yolo8_dir = Path.cwd() / "ultralytics"
sys.path.append(str(yolo8_dir))
print("YOLOv8 repo dodane do PATH:", yolo8_dir.exists())

YOLOv8 repo dodane do PATH: True


In [ ]:
import torch
from ultralytics import YOLO

IMG_SIZE = 640

# 1) Załaduj wewnętrzny model (nano) – to torch.nn.Module
base = YOLO("yolov8n.pt")
nn_model = base.model


# 2) Funkcja testująca forward-pass
def can_run_bs(bs):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    x = torch.zeros(bs, 3, IMG_SIZE, IMG_SIZE, device=device)
    nn_model.to(device).eval()
    try:
        with torch.no_grad():
            _ = nn_model(x)
        torch.cuda.empty_cache()
        return True
    except RuntimeError:
        torch.cuda.empty_cache()
        return False


# 3) Binary search batch
low, high, best = 1, 512, 0
print("Szukam maksymalnego batch_size …")
while low <= high:
    mid = (low + high) // 2
    print(f"  Test bs={mid:<4} … ", end="", flush=True)
    if can_run_bs(mid):
        print("OK")
        best, low = mid, mid + 1
    else:
        print("OOM")
        high = mid - 1

print(f"\n→ Maksymalny batch_size = {best}")

100%|██████████| 6.25M/6.25M [00:00<00:00, 21.4MB/s]


Szukam maksymalnego batch_size …
  Test bs=256  … OK
  Test bs=384  … OK
  Test bs=448  … OK
  Test bs=480  … OK
  Test bs=496  … OK
  Test bs=504  … 

RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


#### Trening

yolo train model=yolov8n.pt data=../../dataset.yaml imgsz=640 batch=256 epochs=25 workers=12 cache=disk name=pothole_yolov8n_cli exist_ok=true

#### Walidacja

python ultralytics/cli.py val weights=ultralytics/runs/train/pothole_yolov8n_cli/weights/best.pt data=../dataset.yaml imgsz=640 name=pothole_yolov8n_cli_val exist_ok=true